# Backpropagation

O material anterior chamou `loss.backward()` a cada iteração sem abrir o que acontece ali dentro. Essa linha calcula a derivada da perda em relação a cada parâmetro do modelo, e é ela que diz ao otimizador em que direção mover cada peso.

O algoritmo que faz isso é a retropropagação, ou backpropagation. Ele não é uma técnica de derivação nova, e sim a aplicação organizada da regra da cadeia,

$$
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial q} \cdot \frac{\partial q}{\partial x}
$$

sobre a sequência de operações que produziu a perda, do fim para o começo. O que torna o algoritmo eficiente é reaproveitar os resultados intermediários em vez de recalcular cada derivada do zero.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(42)

## Grafos computacionais

Toda expressão pode ser escrita como um grafo, em que os nós guardam variáveis e as arestas representam operações. Considere

$$
f(x, y, z) = (x + y) \, z
$$

que se decompõe em duas operações encadeadas, com uma variável intermediária $q$,

$$
q = x + y
\qquad
f = q \, z .
$$

A derivada de $f$ em relação a cada entrada sai da regra da cadeia percorrendo o grafo de trás para frente. Como $\partial f / \partial q = z$ e $\partial q / \partial x = 1$, temos

$$
\frac{\partial f}{\partial x} = z \cdot 1 = z
\qquad
\frac{\partial f}{\partial y} = z \cdot 1 = z
\qquad
\frac{\partial f}{\partial z} = q = x + y .
$$

Com $x = 2$, $y = 3$ e $z = 4$, os três valores esperados são 4, 4 e 5.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)
z = torch.tensor(4.0, requires_grad=True)

q = x + y
f = q * z

f.backward()

print(f"df/dx: {x.grad.item()}")
print(f"df/dy: {y.grad.item()}")
print(f"df/dz: {z.grad.item()}")

Os três valores batem com a conta analítica. O mecanismo que fez isso sozinho é o autograd, e vale entender as três etapas dele em separado.

## Autograd

### Tensores folha

O processo começa nos tensores em relação aos quais queremos derivadas, que são as folhas do grafo. O argumento `requires_grad=True` marca um tensor como folha e faz o PyTorch rastrear toda operação que o envolva. Em um modelo, esses tensores são os pesos e os biases, e o `nn.Module` já os cria assim.

In [ ]:
a = torch.tensor(4.0, requires_grad=True)
b = torch.tensor(5.0, requires_grad=True)
c = torch.tensor(2.0, requires_grad=True)

print(f"a: {a}, requires_grad: {a.requires_grad}")

### Construção do grafo no forward

À medida que as operações são executadas, o PyTorch monta o grafo. Cada tensor produzido por uma operação guarda, em `grad_fn`, a referência à função que o criou e sabe voltar até suas entradas. O grafo é construído a cada passagem para a frente e descartado depois do `backward`.

In [ ]:
u = a * b - c
L = u ** 2

print(f"u: {u.item()}, grad_fn: {u.grad_fn}")
print(f"L: {L.item()}, grad_fn: {L.grad_fn}")

### O backward

A chamada `L.backward()` parte do valor final, que precisa ser um escalar, e percorre o grafo ao contrário aplicando a regra da cadeia. Para $L = u^2$ com $u = ab - c$,

$$
\frac{\partial L}{\partial a} = 2u \, b
\qquad
\frac{\partial L}{\partial b} = 2u \, a
\qquad
\frac{\partial L}{\partial c} = -2u .
$$

Os gradientes calculados são guardados no atributo `.grad` de cada folha.

In [ ]:
L.backward()

print(f"dL/da: {a.grad.item()}, esperado: {2 * 18 * 5}")
print(f"dL/db: {b.grad.item()}, esperado: {2 * 18 * 4}")
print(f"dL/dc: {c.grad.item()}, esperado: {-2 * 18}")

### Acumulação de gradientes

O `backward` soma os novos gradientes aos que já estão em `.grad`, em vez de substituí-los. Repetir a mesma conta sem limpar nada dobra o valor guardado.

In [ ]:
a.grad.zero_()

for iteration in range(2):
    L = (a * b - c) ** 2
    L.backward()
    print(f"depois do backward {iteration + 1}: dL/da = {a.grad.item()}")

Esse comportamento é útil quando um lote grande precisa ser processado em pedaços, somando os gradientes antes de atualizar. Fora esse caso, ele é a razão de o `optimizer.zero_grad()` aparecer em todo laço de treinamento, sempre antes do `backward`.

### Desativação do rastreamento

Construir o grafo custa memória e tempo, e na avaliação não há nada a derivar. O gerenciador de contexto `torch.no_grad()` desliga o rastreamento dentro do seu bloco, e o resultado sai sem `grad_fn`.

In [ ]:
with torch.no_grad():
    L_no_graph = (a * b - c) ** 2
    print(f"requires_grad dentro do bloco: {L_no_graph.requires_grad}")

L_with_graph = (a * b - c) ** 2
print(f"requires_grad fora do bloco: {L_with_graph.requires_grad}")

## Backward analítico de uma MLP

Para ver o que o `backward` executa, vale escrever o percurso por extenso em uma rede pequena. A rede tem uma camada oculta com ReLU, saída linear e erro quadrático médio sobre um lote de $N$ exemplos,

$$
z_1 = X W_1 + b_1
\qquad
h = \mathrm{ReLU}(z_1)
\qquad
\hat{y} = h W_2 + b_2
\qquad
L = \frac{1}{N} \sum_{n} (\hat{y}_n - y_n)^2 .
$$

In [ ]:
X = torch.randn(4, 3)
y = torch.randn(4, 1)

W1 = torch.randn(3, 5, requires_grad=True)
b1 = torch.zeros(5, requires_grad=True)
W2 = torch.randn(5, 1, requires_grad=True)
b2 = torch.zeros(1, requires_grad=True)

z1 = X @ W1 + b1
h = torch.relu(z1)
y_hat = h @ W2 + b2
loss = ((y_hat - y) ** 2).mean()

print(f"perda: {loss.item():.4f}")

Cada camada linear repete o mesmo par de contas: o gradiente do parâmetro é a entrada da camada multiplicada pelo gradiente que chega de cima, e o gradiente que segue para trás é esse mesmo gradiente projetado pelos pesos. O bias soma sobre o lote, e a ReLU repassa o gradiente apenas onde a entrada foi positiva,

$$
\frac{\partial L}{\partial \hat{y}} = \frac{2}{N} (\hat{y} - y)
\qquad
\frac{\partial L}{\partial W_2} = h^{\top} \frac{\partial L}{\partial \hat{y}}
\qquad
\frac{\partial L}{\partial h} = \frac{\partial L}{\partial \hat{y}} \, W_2^{\top}
$$

$$
\frac{\partial L}{\partial z_1} = \frac{\partial L}{\partial h} \odot \mathbf{1}\big[z_1 > 0\big]
\qquad
\frac{\partial L}{\partial W_1} = X^{\top} \frac{\partial L}{\partial z_1}
\qquad
\frac{\partial L}{\partial b_1} = \sum_{n} \frac{\partial L}{\partial z_{1,n}}
$$

Essas expressões são valores, não contas a derivar de novo. Como os intermediários do forward carregam `grad_fn`, usá-los diretamente estenderia o grafo: `detach` devolve um tensor com os mesmos dados, porém fora do rastreamento, e é o equivalente por tensor do `torch.no_grad()`.

In [ ]:
z1_value, h_value, y_hat_value = z1.detach(), h.detach(), y_hat.detach()
N = X.shape[0]

grad_y_hat = 2 * (y_hat_value - y) / N
grad_W2 = h_value.T @ grad_y_hat
grad_b2 = grad_y_hat.sum(dim=0)
grad_h = grad_y_hat @ W2.detach().T
grad_z1 = grad_h * (z1_value > 0)
grad_W1 = X.T @ grad_z1
grad_b1 = grad_z1.sum(dim=0)

loss.backward()

for name, analytic, parameter in [("W1", grad_W1, W1), ("b1", grad_b1, b1),
                                  ("W2", grad_W2, W2), ("b2", grad_b2, b2)]:
    print(f"{name}: coincide com o autograd: {torch.allclose(analytic, parameter.grad)}")

Os quatro gradientes coincidem. A diferença é que o `backward` não conhece a fórmula da rede: cada operação do grafo sabe apenas converter o gradiente da sua saída no gradiente das suas entradas, e a composição dessas etapas locais vale para qualquer expressão.

## Treinamento sem torch.optim

Com as peças no lugar, dá para treinar um modelo sem usar nada do `torch.optim`. O modelo abaixo é um único neurônio, com dois pesos e um bias, e a perda é o erro quadrático em relação a um alvo. A atualização é escrita explicitamente,

$$
\theta \leftarrow \theta - \eta \frac{\partial L}{\partial \theta}
$$

dentro de um bloco `no_grad`, porque a atualização em si não faz parte do cálculo a ser derivado.

In [ ]:
X = torch.tensor([1.2, 0.5])
y_true = torch.tensor(2.0)

W = torch.randn(2, requires_grad=True)
b = torch.randn(1, requires_grad=True)

learning_rate = 0.02
losses = []

In [ ]:
for epoch in range(30):
    prediction = torch.dot(X, W) + b
    loss = (prediction - y_true) ** 2

    loss.backward()

    with torch.no_grad():
        W -= learning_rate * W.grad
        b -= learning_rate * b.grad

        W.grad.zero_()
        b.grad.zero_()

    losses.append(loss.item())
    if epoch % 10 == 0:
        print(f"época {epoch}: perda {loss.item():.4f}")

print(f"perda final: {losses[-1]:.6f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(losses)
plt.xlabel("época")
plt.ylabel("erro quadrático")
plt.grid(True)
plt.show()

As quatro linhas dentro do `no_grad` são exatamente o que o `optimizer.step()` e o `optimizer.zero_grad()` fazem em um treinamento normal. A diferença é que o otimizador guarda a lista de parâmetros e aplica a regra a todos eles, sem que cada um precise ser escrito.

## Gradientes em uma rede

Em uma rede com várias camadas nada muda, exceto o tamanho do grafo. Os parâmetros já vêm com `requires_grad=True`, o `forward` monta o grafo, e um único `backward` a partir da perda preenche o `.grad` de todos eles. Antes desse primeiro `backward` o atributo vale `None`.

In [ ]:
mlp = nn.Sequential(
    nn.Linear(in_features=10, out_features=16),
    nn.ReLU(),
    nn.Linear(in_features=16, out_features=8),
    nn.ReLU(),
    nn.Linear(in_features=8, out_features=2),
)

print(mlp)

In [ ]:
X_batch = torch.randn(5, 10)
y_batch = torch.tensor([0, 1, 1, 0, 1])

criterion = nn.CrossEntropyLoss()
loss = criterion(mlp(X_batch), y_batch)

print(f"perda: {loss.item():.4f}")
print(f"gradiente da última camada antes do backward: {mlp[4].weight.grad}")

In [ ]:
loss.backward()

for index in [0, 2, 4]:
    layer = mlp[index]
    print(f"camada {index}: pesos {tuple(layer.weight.shape)}, gradiente {tuple(layer.weight.grad.shape)}")

Cada gradiente tem exatamente o mesmo formato do parâmetro a que corresponde, porque há uma derivada por peso. É essa correspondência que permite ao otimizador percorrer os parâmetros e atualizá-los um a um, e é dela que trata o próximo material.

## Exercícios

### Exercício 1

Calcule analiticamente as derivadas parciais de $g(x, y) = x^2 y + \sin(x)$ em relação a $x$ e a $y$, avalie as duas em $x = 1$ e $y = 2$, e confira o resultado com o autograd.

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
y = torch.tensor(2.0, requires_grad=True)

### Exercício 2

Treine o neurônio da seção anterior com três taxas de aprendizado, uma muito menor, uma próxima e uma muito maior que a usada aqui, e desenhe as três curvas de perda no mesmo gráfico. O que acontece com a maior delas?

In [ ]:
learning_rates = []

### Exercício 3

Substitua a atualização explícita por um `torch.optim.SGD` com a mesma taxa de aprendizado, mantendo tudo o mais igual, e verifique que as curvas de perda coincidem.

In [ ]:
# optimizer = torch.optim.SGD(...)